# 1008 26일차

## 1. PyPDFLoader

PDF 파일을 쪽마다 `Document` 하나로 불러오는 로더 (`pip install pypdf`)

PDF는 텍스트 파일이 아니라서 `TextLoader`(day24 RAG §1)로는 읽을 수 없음. `PyPDFLoader`는 쪽마다 글자만 뽑아 `page_content`에 담음

1. `PyPDFLoader(경로)` : PDF 경로로 로더를 만듦
2. `load()` : 쪽마다 `Document` 하나씩 담은 리스트를 돌려줌. N쪽 PDF → `Document` N개

- PDF에서 줄이 바뀌는 곳은 `page_content`에 `\n`으로 들어감
- 쪽 번호·머리말처럼 쪽에 찍힌 글자는 본문이 아니어도 `page_content`에 같이 들어감

### 1-1. metadata

쪽마다 `metadata`에 파일 경로·쪽 번호와 PDF 파일 정보가 들어감

| 키 | 내용 |
|---|---|
| `source` | 파일 경로 |
| `total_pages` | 전체 쪽 수 |
| `page` | 0부터 센 쪽 번호. 첫 쪽이 `0` |
| `page_label` | PDF에 정해 둔 쪽 번호 (문자열). 없으면 1부터 세서 첫 쪽이 `'1'` |
| 그 외 | PDF 파일 정보 (`producer`·`creator`·`creationdate`·`title` 등) |

- 한 PDF 안에서는 `page`·`page_label`만 쪽마다 다르고 나머지 값은 같음
- 파일 정보는 PDF를 만들 때 적힌 값이라 PDF마다 다름

### 1-2. 분할

`split_documents(pdf_docs)` : `load()`로 받은 쪽 리스트를 청크로 나눔 (분할 규칙은 day24 RAG §1-1)

쪽 하나에는 여러 내용이 섞여 있어서 쪽을 그대로 임베딩하면 질문과 관련 있는 부분을 좁힐 수 없음

- 청크마다 원래 쪽의 `metadata`가 그대로 붙어서 `page`로 몇 쪽에서 나온 청크인지 알 수 있음
- 쪽마다 따로 나눠서 청크 하나가 두 쪽에 걸치지 않음
- `load_and_split(splitter)`(day24 RAG §1)는 안에서 `load()`를 다시 불러 PDF를 한 번 더 읽음. 이미 불러 둔 `pdf_docs`가 있으면 `split_documents`로 나눔

## 2. HuggingFaceEmbeddings

Hugging Face Hub에 공개된 임베딩 모델을 내 컴퓨터에서 돌리는 LangChain 클래스 (`pip install langchain-huggingface sentence-transformers`)

API를 부르지 않아서 키·크레딧이 들지 않음. 모델 파일을 한 번 받아 두면 인터넷 없이도 임베딩할 수 있음

1. `model_name` : Hub 저장소 이름 (`"BAAI/bge-m3"`, `"Qwen/Qwen3-Embedding-0.6B"`)
2. `model_kwargs` : 모델을 불러올 때 넘기는 인자. `"device"`는 `"cpu"`·`"cuda"`, `"local_files_only": True`면 받아 둔 파일만 씀
3. `embed_query`·`embed_documents` : `OpenAIEmbeddings`와 같은 메서드 (day24 RAG §2)

| 모델 | 차원 | 가중치 파일 |
|---|---|---|
| `BAAI/bge-m3` | 1024 | `pytorch_model.bin` 2.27GB |
| `Qwen/Qwen3-Embedding-0.6B` | 1024 | `model.safetensors` 1.19GB |

- 처음 실행할 때 가중치·토크나이저·설정 파일을 `~/.cache/huggingface/hub`에 받음. 다음 실행부터는 받아 둔 파일을 읽음
- 임베딩 계산은 `device`에서 함. CPU에서는 청크가 많을수록 저장(`from_documents`)이 오래 걸림
- `OpenAIEmbeddings` 자리에 넣어도 FAISS 저장·불러오기와 RAG 체인 코드는 그대로 (day25 RAG §2, §4)

#### cf) 질문 지시문

질문 앞에 지시문을 붙여 학습된 임베딩 모델은 질문을 임베딩할 때도 지시문을 붙여야 함 (Qwen3-Embedding)

- 지시문은 모델 설정 파일(`config_sentence_transformers.json`)의 `prompts`에 `query`로 들어 있음. bge-m3는 `prompts`가 없음
- `query_encode_kwargs={"prompt_name": "query"}` : `embed_query`에만 지시문을 붙임. 저장할 청크(`embed_documents`)에는 안 붙음
- `encode_kwargs`에 넣으면 청크에도 붙음
- 지시문 없이 임베딩해도 에러는 없고 검색 성능만 조금 떨어짐 (모델 카드 기준 1~5%)

## 3. 임베딩 모델 교체

**DB를 저장한 임베딩 모델과 질문을 임베딩하는 모델이 같아야 함** (day24 RAG §3, day25 RAG §4)

모델마다 벡터 공간이 달라서, 다른 모델로 만든 질문 벡터와 청크 벡터는 거리를 재도 뜻이 비슷한지 알 수 없음. 모델을 바꾸면 청크를 새 모델로 다시 임베딩해 저장해야 함

1. 차원이 다를 때 : `load_local`은 차원을 확인하지 않아서 불러오기는 됨. 검색할 때 FAISS에서 `AssertionError`
2. 차원이 같을 때 : 에러 없이 질문과 상관없는 청크가 나옴. 채팅 모델은 프롬프트대로 "주어진 정보로는 답변할 수 없습니다."라고 답함 (day25 RAG §1 cf)

- `text-embedding-3-small`(1536차원)로 저장한 DB를 bge-m3(1024차원)로 검색하면 1번
- bge-m3와 Qwen3-Embedding-0.6B는 둘 다 1024차원이라 서로 바꿔 쓰면 2번
- 모델을 바꿀 때는 새 폴더에 다시 저장하고, 저장 코드와 챗봇 코드의 `model_name`을 같이 바꿈

## 4. 요약·결론 질문

"요약해줘", "결론이 뭐야?"처럼 문서 전체나 섹션 이름을 묻는 질문은 벡터 검색으로 답이 든 청크를 찾기 어려움

벡터 검색은 질문과 내용이 비슷한 청크를 찾음. 이런 질문에는 비교할 내용이 없어서 초록·결론 청크가 위로 오기 어려움

1. 내용을 넣어 묻기 : "앞으로 어떤 연구를 할 계획이야?"처럼 알고 싶은 내용을 넣으면 그 내용이 든 청크가 나옴
2. 본문이 아닌 쪽 빼기 : 참고문헌·그림 속 글자처럼 문장이 아닌 청크가 이런 질문의 1·2위를 차지하기도 함. 저장 전에 그 쪽을 빼면 본문 청크가 대신 나옴

- `k`를 늘려도 순위가 한참 밀린 청크는 들어오지 않음
- 본문이 아닌 쪽을 빼도 초록·결론 청크가 위로 온다는 보장은 없음
- 같은 질문이라도 임베딩 모델마다 1·2위로 나오는 청크가 다를 수 있음

#### cf) 답이 나와도 검색이 틀렸을 수 있음

넘어간 청크에 답이 없어도 채팅 모델이 답을 만들 때가 있음 (day25 RAG §1 cf)

- 제목·저자만 든 청크가 넘어가도 학습 데이터에 있는 유명한 문서면 요약이 나올 수 있음
- 검색이 맞았는지는 `response['context']`에 들어온 청크로 확인함 (day25 RAG §2-1)
- 임베딩 모델을 바꿔서 비교할 때도 답 대신 `context`를 비교함